In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/python-functions-with-docstrings/python_functions_and_documentation_dataset.csv
/kaggle/input/bpe-encoded-tokens/bpe_joint.merges
/kaggle/input/bpe-encoded-tokens/bpe_code.vocab.json
/kaggle/input/bpe-encoded-tokens/bpe_joint.vocab.json
/kaggle/input/bpe-encoded-tokens/bpe_code.meta.json
/kaggle/input/bpe-encoded-tokens/bpe_joint.meta.json
/kaggle/input/bpe-encoded-tokens/bpe_doc.vocab.json
/kaggle/input/bpe-encoded-tokens/bpe_doc.merges
/kaggle/input/bpe-encoded-tokens/bpe_code.merges
/kaggle/input/bpe-encoded-tokens/bpe_doc.meta.json


In [11]:
# ================================
# NOTEBOOK B: Word2Vec Training & Evaluation
# Task 4 & 5 (Project Spec)
# ================================
!pip install gensim

import os, json, random, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from gensim.models import Word2Vec

warnings.filterwarnings("ignore")

# Reproducibility
SEED = 42
random.seed(SEED); np.random.seed(SEED)

# Output folder
OUTPUT_DIR = "word2vec"
os.makedirs(OUTPUT_DIR, exist_ok=True)


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 38.6/38.6 MB 49.2 MB/s eta 0:00:00:00:0100:01
  Attempting uninstall: scipy
    Found existing installation: scipy 1.15.3
    Uninstalling scipy-1.15.3:
      Successfully uninstalled scipy-1.15.3
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cesium 0.12.4 requires numpy<3.0,>=2.0, but you have numpy 1.26.4 which is incompatible.
tsfresh 0.21.0 requires scipy>=1.14.0; python_version >= "3.10", but you have scipy 1.13.1 which is incompatible.
dopamine-rl 4.1.2 requires gymnasium>=1.0.0, but you have gymnasium 0.29.0 which is incompatible.
imbalanced-learn 0.13.0 requires scikit-learn<2,>=1.3.2, but you have scikit-learn 1.2.2 which is incompatible.
plotnine 0.14.5 requires matplotlib>=3.8.0, but you have matplotlib 3.7.2 which is incompat

In [3]:
# Paths
CSV_PATH = "/kaggle/input/python-functions-with-docstrings/python_functions_and_documentation_dataset.csv"
BPE_DIR  = "/kaggle/input/bpe-encoded-tokens"

CODE_VOCAB  = os.path.join(BPE_DIR, "bpe_code.vocab.json")
DOC_VOCAB   = os.path.join(BPE_DIR, "bpe_doc.vocab.json")
JOINT_VOCAB = os.path.join(BPE_DIR, "bpe_joint.vocab.json")

# Load dataset (limit to manageable size for Kaggle runtime)
df = pd.read_csv(CSV_PATH).head(5000)
print("Dataset loaded:", df.shape)

# Load BPE vocabs (for reference only)
with open(CODE_VOCAB) as f: code_vocab = json.load(f)
with open(DOC_VOCAB) as f:  doc_vocab  = json.load(f)
with open(JOINT_VOCAB) as f: joint_vocab = json.load(f)

print("Vocab sizes → Code:", len(code_vocab), "Doc:", len(doc_vocab), "Joint:", len(joint_vocab))


Dataset loaded: (5000, 13)
Vocab sizes → Code: 362 Doc: 364 Joint: 421


In [4]:
def prepare_sentences(df, mode="combined"):
    sents = []
    for _, row in df.iterrows():
        if mode == "code":
            toks = str(row["code"]).split()
        elif mode == "doc":
            toks = str(row["docstring"]).split()
        else:  # combined
            toks = (str(row["code"]) + " " + str(row["docstring"])).split()
        if toks: sents.append(toks)
    return sents

code_sents = prepare_sentences(df, "code")
doc_sents  = prepare_sentences(df, "doc")
comb_sents = prepare_sentences(df, "combined")

print(f"Sentences → Code={len(code_sents)}, Doc={len(doc_sents)}, Combined={len(comb_sents)}")


Sentences → Code=5000, Doc=5000, Combined=5000


In [5]:
models = {}
for name, sents in [("code", code_sents), ("doc", doc_sents), ("combined", comb_sents)]:
    print(f"\nTraining {name} model...")
    w2v = Word2Vec(
        sentences=sents,
        vector_size=100,
        window=5,
        min_count=5,
        sg=1,             # Skip-gram
        workers=4,
        epochs=10
    )
    w2v.save(f"{OUTPUT_DIR}/w2v_{name}.model")
    models[name] = w2v

print("✅ All Word2Vec models trained & saved.")



Training code model...

Training doc model...

Training combined model...
✅ All Word2Vec models trained & saved.


In [6]:
def eval_similarity(model, max_words=50):
    vocab = list(model.wv.key_to_index.keys())
    sims=[]
    for w in vocab[:max_words]:
        sims.extend(model.wv.most_similar(w, topn=3))
    avg = np.mean([s for _,s in sims]) if sims else 0
    return {"avg_similarity": avg, "examples": sims[:10]}

similarity_results = {n: eval_similarity(m) for n,m in models.items()}


In [7]:
def eval_completion(model, df, mode="code"):
    scores=[]
    for _, row in df.sample(min(100,len(df))).iterrows():
        toks = str(row['code']).split() if mode=="code" else str(row['docstring']).split()
        if len(toks)<2: continue
        ctx, target = toks[:-1], toks[-1]
        try:
            pred = model.wv.most_similar(positive=ctx, topn=1)[0][0]
            scores.append(1 if pred==target else 0)
        except: pass
    return {"accuracy": np.mean(scores) if scores else 0, "tests": len(scores)}

completion_results = {
    "code": eval_completion(models["code"], df,"code"),
    "doc":  eval_completion(models["doc"], df,"doc"),
    "combined": eval_completion(models["combined"], df,"code")
}


In [8]:
def eval_relevance(model, df):
    scores=[]
    for _, row in df.sample(min(50,len(df))).iterrows():
        code_t = str(row['code']).split()[:5]
        doc_t  = str(row['docstring']).split()[:5]
        for c in code_t:
            for d in doc_t:
                try: scores.append(model.wv.similarity(c,d))
                except: scores.append(0)
    return {"avg_relevance": np.mean(scores) if scores else 0, "tests": len(scores)}

relevance_results = {"combined": eval_relevance(models["combined"], df)}


In [9]:
def plot_embeddings(model, name):
    words = list(model.wv.key_to_index.keys())[:100]
    vecs  = np.array([model.wv[w] for w in words])
    if len(vecs) < 2: return
    pca = PCA(n_components=2).fit_transform(vecs)
    plt.figure(figsize=(10,8))
    plt.scatter(pca[:,0], pca[:,1])
    for i,w in enumerate(words[:20]):
        plt.annotate(w,(pca[i,0],pca[i,1]))
    plt.title(f"Word2Vec PCA - {name}")
    plt.savefig(f"{OUTPUT_DIR}/pca_{name}.png")
    plt.close()

for n,m in models.items():
    plot_embeddings(m,n)
print("✅ PCA visualizations created.")


✅ PCA visualizations created.


In [10]:
results = {
    "similarity": similarity_results,
    "completion": completion_results,
    "relevance": relevance_results
}

with open(f"{OUTPUT_DIR}/results.json","w") as f:
    json.dump(results,f,indent=2)

print("✅ Results saved. Summary:")
print(json.dumps(results, indent=2))


✅ Results saved. Summary:
{
  "similarity": {
    "code": {
      "avg_similarity": 0.6085781677563985,
      "examples": [
        [
          "left_indexer,",
          0.6615521907806396
        ],
        [
          "right_indexer",
          0.6516517400741577
        ],
        [
          "random.randint(0,",
          0.6415725350379944
        ],
        [
          "frequent",
          0.6985722780227661
        ],
        [
          "or,",
          0.695268452167511
        ],
        [
          "keeping",
          0.69477778673172
        ],
        [
          "self.gen_mode",
          0.7055620551109314
        ],
        [
          "self._selection",
          0.697912871837616
        ],
        [
          "not",
          0.6938601136207581
        ],
        [
          "self.values)",
          0.6389839053153992
        ]
      ]
    },
    "doc": {
      "avg_similarity": 0.6141858957211177,
      "examples": [
        [
          "units",
          0.6368